<a href="https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os

# Clone the repository to access the dataset
!git clone https://github.com/nomanamir20/flyrank-ml-internship.git

# Change to the project directory if necessary, or ensure paths are relative to /content/
# os.chdir('/content/flyrank-ml-internship')

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. Question

## Research Question

Can a machine-learning ranking model identify keyword-article content rows that should be prioritized for human review based on observed SEO performance signals?

## Problem

Content teams may have many keyword-article pages to review, making it difficult to decide which pages should be inspected first. The goal of this project is to use observed content-performance features to produce a ranked review queue that helps humans focus their attention on higher-priority rows.

## Decision Supported

The model supports one specific decision:

> **Which keyword-article rows should a human reviewer inspect first?**

The output is therefore a prioritization score and ranked queue rather than an automatic content-change decision.

## Success Criteria

The model is evaluated primarily as a ranking system using:

- Precision@20
- Precision@50
- Precision@100
- ROC-AUC
- Average Precision

The analysis also compares the learned model with a transparent baseline on the same validation design.

## Boundary of the Question

This project does **not** attempt to predict Google's ranking algorithm, establish causal relationships, or automatically decide what content should be published, rewritten, deleted, redirected, or changed.

The model output is treated as **decision-support** based on observed and measured signals. Human review remains necessary before any content action.

In [11]:
# Section 1: Question
# Define the research question and decision-support boundary.

research_question = (
    "Can a machine-learning ranking model identify keyword-article "
    "content rows that should be prioritized for human review based "
    "on observed SEO performance signals?"
)

decision_supported = (
    "Prioritize keyword-article rows for human review."
)

success_metrics = [
    "Precision@20",
    "Precision@50",
    "Precision@100",
    "ROC-AUC",
    "Average Precision",
]

boundaries = [
    "Decision-support only",
    "No causal claims",
    "No prediction of Google's ranking algorithm",
    "No automatic publishing",
    "No automatic rewriting",
    "No automatic deletion",
    "Human review required before content action",
]

print("RESEARCH QUESTION")
print(research_question)

print("\nDECISION SUPPORTED")
print(decision_supported)

print("\nSUCCESS METRICS")
for metric in success_metrics:
    print("-", metric)

print("\nBOUNDARIES")
for boundary in boundaries:
    print("-", boundary)

RESEARCH QUESTION
Can a machine-learning ranking model identify keyword-article content rows that should be prioritized for human review based on observed SEO performance signals?

DECISION SUPPORTED
Prioritize keyword-article rows for human review.

SUCCESS METRICS
- Precision@20
- Precision@50
- Precision@100
- ROC-AUC
- Average Precision

BOUNDARIES
- Decision-support only
- No causal claims
- No prediction of Google's ranking algorithm
- No automatic publishing
- No automatic rewriting
- No automatic deletion
- Human review required before content action


## 2. Data

This study uses the anonymized FlyRank internship starter dataset, which contains 30,000 rows and 44 columns. Each row represents a pseudonymized content item with SEO and content-performance measurements.

For this analysis, the dataset was restricted to the keyword-article lane, resulting in 27,207 rows. The analysis focuses on identifying keyword-article rows that should be prioritized for human review based on their observed decline risk.

The dataset contains numeric and categorical SEO/content features. The target variable is `is_declining_label`. The variables `trend_direction` and `trend_pct` were excluded because the label is derived from the trend information, creating a direct leakage risk. The pseudonymous identifiers `client_id` and `content_id` were also excluded from the model features and used only for grouping, joining, and validation purposes.

The dataset is anonymized and contains no client names, domains, URLs, private queries, or other directly identifying information.

The analysis uses a client-grouped validation design so that validation clients do not overlap with training clients. This provides a stricter test of whether the ranking approach generalizes across clients.

### Data scope

- Starter dataset: 30,000 rows × 44 columns
- Keyword-article analysis lane: 27,207 rows
- Numeric model features: 22
- Categorical model features: 8
- Total model features: 30
- Overall keyword-article decline rate: 56.1%

### Exclusions

The following variables were excluded from model features:

- `trend_direction` — excluded because it contributes directly to the label definition.
- `trend_pct` — excluded for the same leakage concern.
- `is_declining_label` — target variable, not a feature.
- `client_id` — retained only for client-grouped validation.
- `content_id` — retained only as an identifier and not used as a predictive feature.

The analysis is therefore based on public-safe, anonymized data and is intended for directional decision-support rather than production prediction.

In [12]:
import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Locate and load the anonymized starter dataset
# ------------------------------------------------------------

possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
]

data_path = None

for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Check that the repository is mounted/cloned and the dataset path is correct."
    )

df_raw = pd.read_csv(data_path)

# Derive 'is_declining_label' from 'trend_direction'
df_raw['is_declining_label'] = (df_raw['trend_direction'] == 'down').astype(int)

print("DATASET LOADED")
print("-" * 50)
print(f"Path: {data_path}")
print(f"Dataset shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")


# ------------------------------------------------------------
# 2. Basic dataset checks
# ------------------------------------------------------------

required_columns = [
    "client_id",
    "content_id",
    "content_type",
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

missing_columns = [
    col for col in required_columns
    if col not in df_raw.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

print("\nCOLUMN CHECK")
print("-" * 50)
print("Required columns found:", True)


# ------------------------------------------------------------
# 3. Restrict analysis to keyword-article rows
# ------------------------------------------------------------

keyword_article_values = [
    "keyword_article",
    "keyword article",
    "keyword-article"
]

content_type_normalized = (
    df_raw["content_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

keyword_mask = content_type_normalized.isin(keyword_article_values)

df = df_raw.loc[keyword_mask].copy()

if len(df) == 0:
    # Fallback: inspect the available content types
    print("\nAvailable content_type values:")
    print(df_raw["content_type"].value_counts(dropna=False))
    raise ValueError(
        "No keyword-article rows were found. "
        "Check the exact content_type value used in the dataset."
    )

print("\nKEYWORD-ARTICLE LANE")
print("-" * 50)
print(f"Keyword article lane shape: {df.shape}")


# ------------------------------------------------------------
# 4. Target distribution
# ------------------------------------------------------------

target_col = "is_declining_label"

target_distribution = (
    df[target_col]
    .value_counts(dropna=False)
    .sort_index()
)

decline_rate = df[target_col].mean()

print("\nTARGET DISTRIBUTION")
print("-" * 50)
print(target_distribution)

print(f"\nOverall keyword-article decline rate: {decline_rate:.4f}")
print(f"Overall keyword-article decline rate: {decline_rate:.2%}")


# ------------------------------------------------------------
# 5. Define excluded columns
# ------------------------------------------------------------

excluded_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]

missing_excluded = [
    col for col in excluded_columns
    if col not in df.columns
]

if missing_excluded:
    raise ValueError(
        f"Expected excluded columns are missing: {missing_excluded}"
    )


# ------------------------------------------------------------
# 6. Build model feature list
# ------------------------------------------------------------

feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]

X = df[feature_columns].copy()
y = df[target_col].copy()

numeric_features = X.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\nFEATURE SUMMARY")
print("-" * 50)
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Total model features: {len(feature_columns)}")

print("\nEXCLUDED VARIABLES")
print("-" * 50)

for col in excluded_columns:
    print(f"{col} : excluded")


# ------------------------------------------------------------
# 7. Missingness summary
# ------------------------------------------------------------

missing_summary = (
    X.isna()
    .mean()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[
    missing_summary > 0
]

print("\nMISSINGNESS CHECK")
print("-" * 50)

if len(missing_summary) == 0:
    print("No missing values found in model features.")
else:
    print(
        missing_summary.head(10)
        .to_string()
    )


# ------------------------------------------------------------
# 8. Client information for grouped validation
# ------------------------------------------------------------

client_count = df["client_id"].nunique()
content_count = df["content_id"].nunique()

print("\nIDENTIFIER SUMMARY")
print("-" * 50)
print(f"Unique clients: {client_count}")
print(f"Unique content items: {content_count}")


# ------------------------------------------------------------
# 9. Public-safety checks
# ------------------------------------------------------------

private_columns = [
    "client_name",
    "domain",
    "url",
    "query",
    "keyword"
]

present_private_columns = [
    col for col in private_columns
    if col in df.columns
]

print("\nPUBLIC-SAFETY CHECK")
print("-" * 50)

if len(present_private_columns) == 0:
    print("No known private-name, domain, URL, query, or keyword columns detected.")
else:
    print(
        "Potential private columns detected:",
        present_private_columns
    )


# ------------------------------------------------------------
# 10. Store reusable variables for later sections
# ------------------------------------------------------------

data_summary = {
    "dataset_rows": int(df_raw.shape[0]),
    "dataset_columns": int(df_raw.shape[1]),
    "keyword_article_rows": int(df.shape[0]),
    "keyword_article_columns": int(df.shape[1]),
    "decline_rate": float(decline_rate),
    "numeric_features": int(len(numeric_features)),
    "categorical_features": int(len(categorical_features)),
    "total_features": int(len(feature_columns)),
    "unique_clients": int(client_count),
    "unique_content_items": int(content_count)
}

print("\nSECTION 2 DATA SUMMARY")
print("-" * 50)

for key, value in data_summary.items():
    print(f"{key}: {value}")

DATASET LOADED
--------------------------------------------------
Path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Dataset shape: (30000, 45)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']

COLUMN CHECK
--------------

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [13]:
# ============================================================
# COLAB SETUP — RESTORE FLYRANK REPOSITORY
# ============================================================

import os
import subprocess

REPO_URL = "https://github.com/nomanamir20/flyrank-ml-internship.git"
REPO_PATH = "/content/flyrank-ml-internship"

print("=" * 60)
print("RESTORING FLYRANK REPOSITORY")
print("=" * 60)

# Check whether repository already exists
if os.path.exists(REPO_PATH):
    print("\nRepository already exists:")
    print(REPO_PATH)
else:
    print("\nRepository not found.")
    print("Cloning repository...")

    subprocess.run(
        ["git", "clone", REPO_URL, REPO_PATH],
        check=True
    )

    print("\nRepository cloned successfully.")


# ------------------------------------------------------------
# FIND DATASET
# ------------------------------------------------------------

DATA_PATH = os.path.join(
    REPO_PATH,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

print("\nDATASET CHECK")
print("-" * 60)
print("Expected dataset:")
print(DATA_PATH)

if os.path.exists(DATA_PATH):
    print("\nPASS: Dataset found.")

    import pandas as pd

    test_df = pd.read_csv(DATA_PATH)

    print("Dataset shape:", test_df.shape)
    print("Dataset columns:", len(test_df.columns))

else:
    print("\nERROR: Dataset was not found at the expected path.")

    print("\nSearching repository for CSV files...")

    csv_files = []

    for root, dirs, files in os.walk(REPO_PATH):
        for file in files:
            if file.lower().endswith(".csv"):
                csv_files.append(
                    os.path.join(root, file)
                )

    if csv_files:
        print("\nCSV files found:")
        for file in csv_files:
            print(" -", file)
    else:
        print("No CSV files found in the repository.")

        raise FileNotFoundError(
            "The repository exists, but the required dataset CSV "
            "could not be found."
        )

print("\n" + "=" * 60)
print("COLAB SETUP COMPLETED")
print("=" * 60)

RESTORING FLYRANK REPOSITORY

Repository already exists:
/content/flyrank-ml-internship

DATASET CHECK
------------------------------------------------------------
Expected dataset:
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

PASS: Dataset found.
Dataset shape: (30000, 44)
Dataset columns: 44

COLAB SETUP COMPLETED


In [14]:
# ============================================================
# SECTION 3 — METHODOLOGY
# Assumptions, features, label definition, baseline,
# validation design, leakage checks
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 60)
print("METHODOLOGY")
print("=" * 60)

# ------------------------------------------------------------
# 1. LOAD RAW DATA
# ------------------------------------------------------------

REPO_PATH = "/content/flyrank-ml-internship"
DATA_PATH = os.path.join(
    REPO_PATH,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at:\n{DATA_PATH}\n\n"
        "Restore/clone the FlyRank repository in Colab first."
    )

df = pd.read_csv(DATA_PATH)

print("\nDATA LOADED")
print("-" * 60)
print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. REQUIRED RAW COLUMNS
# ------------------------------------------------------------

required_raw_columns = [
    "content_id",
    "client_id",
    "content_type",
    "trend_direction",
    "trend_pct"
]

missing_required = [
    col for col in required_raw_columns
    if col not in df.columns
]

if missing_required:
    raise ValueError(
        f"Required raw columns are missing: {missing_required}"
    )

print("Required raw columns found: True")

# ------------------------------------------------------------
# 3. RECREATE TARGET LABEL
# ------------------------------------------------------------
# IMPORTANT:
# The starter dataset does NOT contain is_declining_label.
# The label is derived from trend_direction.
#
# Positive class:
#   1 = declining
#
# Negative class:
#   0 = not declining

def create_decline_label(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("declining")
        .astype(int)
    )

df["is_declining_label"] = create_decline_label(
    df["trend_direction"]
)

print("\nLABEL CREATED")
print("-" * 60)
print("Target variable: is_declining_label")
print("Positive class: 1 = declining")
print("Negative class: 0 = not declining")

print("\nTarget distribution:")
print(
    df["is_declining_label"]
    .value_counts()
    .sort_index()
)

print(
    "\nOverall decline rate: "
    f"{df['is_declining_label'].mean():.4f}"
)

# ------------------------------------------------------------
# 4. KEYWORD-ARTICLE LANE
# ------------------------------------------------------------

keyword_article_values = [
    "keyword_article",
    "keyword article",
    "keyword-article"
]

keyword_article_mask = (
    df["content_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(keyword_article_values)
)

lane_df = df.loc[keyword_article_mask].copy()

print("\nKEYWORD-ARTICLE LANE")
print("-" * 60)
print("Keyword article shape:", lane_df.shape)

if len(lane_df) == 0:
    raise ValueError(
        "No keyword_article rows were found. "
        "Check the content_type values."
    )

print(
    "Keyword-article decline rate: "
    f"{lane_df['is_declining_label'].mean():.4f}"
)

# ------------------------------------------------------------
# 5. FEATURE DEFINITIONS
# ------------------------------------------------------------

target_column = "is_declining_label"

excluded_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]

feature_columns = [
    col for col in lane_df.columns
    if col not in excluded_columns
]

# Identify feature types
numeric_features = [
    col for col in feature_columns
    if pd.api.types.is_numeric_dtype(lane_df[col])
]

categorical_features = [
    col for col in feature_columns
    if col not in numeric_features
]

print("\nFEATURE DEFINITION")
print("-" * 60)

print("Target:", target_column)

print("\nExcluded variables:")
for col in excluded_columns:
    print(f" - {col}")

print("\nNumeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total model features:", len(feature_columns))

# ------------------------------------------------------------
# 6. LEAKAGE CHECK
# ------------------------------------------------------------

print("\nLEAKAGE CHECK")
print("-" * 60)

leakage_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]

leakage_present_in_features = [
    col for col in leakage_columns
    if col in feature_columns
]

if leakage_present_in_features:
    raise ValueError(
        "Potential leakage/identifier columns found in features: "
        f"{leakage_present_in_features}"
    )

print("PASS: No target, trend, client ID, or content ID")
print("columns are included as model features.")

# ------------------------------------------------------------
# 7. TARGET / FEATURE SANITY CHECK
# ------------------------------------------------------------

print("\nSANITY CHECK")
print("-" * 60)

print("Rows in keyword-article lane:", len(lane_df))
print("Target missing values:", lane_df[target_column].isna().sum())
print("Feature count:", len(feature_columns))

if lane_df[target_column].isna().sum() > 0:
    raise ValueError(
        "Target contains missing values."
    )

# ------------------------------------------------------------
# 8. CLIENT GROUPING CHECK
# ------------------------------------------------------------

unique_clients = lane_df["client_id"].nunique()
unique_content = lane_df["content_id"].nunique()

print("\nGROUPING / IDENTIFIER CHECK")
print("-" * 60)
print("Unique clients:", unique_clients)
print("Unique content items:", unique_content)

print(
    "\nClient IDs are retained only for grouped validation "
    "and are excluded from model features."
)

print(
    "Content IDs are retained only as identifiers "
    "and are excluded from model features."
)

# ------------------------------------------------------------
# 9. MISSINGNESS SUMMARY
# ------------------------------------------------------------

missingness = (
    lane_df[feature_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("\nMISSINGNESS CHECK")
print("-" * 60)

missingness_nonzero = missingness[missingness > 0]

if len(missingness_nonzero) == 0:
    print("No missing feature values detected.")
else:
    print(
        missingness_nonzero.head(15).to_string()
    )

# ------------------------------------------------------------
# 10. METHODOLOGY SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("METHODOLOGY SUMMARY")
print("=" * 60)

print("""
ASSUMPTIONS
1. The task is treated as binary classification and ranking.
2. The model is used for decision-support and human review,
   not automatic content changes.
3. Rows represent keyword-article content items in the
   selected analysis lane.
4. Client IDs are used for grouped validation only and are
   not model features.
5. Content IDs are identifiers only and are not model features.
6. The observed label is treated as an evaluation outcome,
   not as a causal explanation.

LABEL DEFINITION
Target variable: is_declining_label
Positive class: 1 = declining
Negative class: 0 = not declining

LEAKAGE CONTROLS
- trend_direction excluded
- trend_pct excluded
- is_declining_label excluded
- client_id excluded
- content_id excluded

VALIDATION DESIGN
Client-grouped validation is used so validation clients do
not overlap with training clients.

MODEL USE
The resulting score is a prioritisation signal for human review.
It is not treated as causal evidence and does not automatically
trigger content changes.
""")

print("\nSECTION 3 COMPLETED SUCCESSFULLY")


METHODOLOGY

DATA LOADED
------------------------------------------------------------
Dataset shape: (30000, 44)
Required raw columns found: True

LABEL CREATED
------------------------------------------------------------
Target variable: is_declining_label
Positive class: 1 = declining
Negative class: 0 = not declining

Target distribution:
is_declining_label
0    30000
Name: count, dtype: int64

Overall decline rate: 0.0000

KEYWORD-ARTICLE LANE
------------------------------------------------------------
Keyword article shape: (27207, 45)
Keyword-article decline rate: 0.0000

FEATURE DEFINITION
------------------------------------------------------------
Target: is_declining_label

Excluded variables:
 - trend_direction
 - trend_pct
 - is_declining_label
 - client_id
 - content_id

Numeric features: 29
Categorical features: 11
Total model features: 40

LEAKAGE CHECK
------------------------------------------------------------
PASS: No target, trend, client ID, or content ID
columns 

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
